## **Finland labor market gig shift 2019-2026**

In [1]:
import numpy as np
import pandas as pd

## 1. Traficom Taxi Licences (2016–2023)
Source: Finnish Transport and Communications Agency (Traficom)
Issues to fix:
- Semicolon separator instead of comma
- Thousands separator spaces in licence counts e.g. "9 624" → 9624
- Date column needs splitting into year and month

In [2]:
df_taxi = pd.read_csv('../data/raw/traficom_taxi_licences_2016_2023.csv', sep=';')

In [4]:
df_taxi.shape

(9, 2)

In [5]:
df_taxi.head()

,Date,Number of licenses
0,12/2016,9 624
1,12/2017,9 499
2,6/2018,9 487
3,12/2018,11 852
4,12/2019,12 332


In [6]:
df_taxi.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Date                9 non-null      str  
 1   Number of licenses  9 non-null      str  
dtypes: str(2)
memory usage: 276.0 bytes


In [8]:
df_taxi.isnull().sum()

Date                  0
Number of licenses    0
dtype: int64

In [13]:
# Fix column names to lowercase with underscores
df_taxi.columns = ['date', 'number_of_licences']

In [ ]:
# Convert date to datetime
df_taxi['date'] = pd.to_datetime(df_taxi['date'], format='%m/%Y')

In [15]:
df_taxi['date'].dtype

dtype('<M8[us]')

In [16]:
# Remove thousands separator spaces and convert to integer
df_taxi['number_of_licences'] = df_taxi['number_of_licences'].str.replace(' ', '').astype(int)

In [18]:
# Extract year and month as separate columns
df_taxi['year'] =  df_taxi['date'].dt.year
df_taxi['month']= df_taxi['date'].dt.month

In [19]:
# Add source column for traceability in PostgreSQL later
df_taxi['source'] = 'traficom'

In [20]:
df_taxi.head()

,date,number_of_licences,year,month,source
0,2016-12-01,9624,2016,12,traficom
1,2017-12-01,9499,2017,12,traficom
2,2018-06-01,9487,2018,6,traficom
3,2018-12-01,11852,2018,12,traficom
4,2019-12-01,12332,2019,12,traficom


In [23]:
df_taxi.dtypes

date                  datetime64[us]
number_of_licences             int64
year                           int32
month                          int32
source                           str
dtype: object

In [24]:
df_taxi.to_csv('../data/cleaned/traficom_taxi_licences_2016_2023.csv', index=False)
print("Exported Succesfully")

Exported Succesfully


## 2. StatFin Population by Activity Type (1987–2024)
Source: Statistics Finland (stat.fi)
Issues to fix:
- Wide format (years as columns) needs reshaping to long format (one row per year)
- First row is a header artifact from stat.fi export
- Values are percentages of total population

In [56]:
df_population = pd.read_csv('../data/raw/statfin_population_by_activity_1987_2024.csv', skiprows=1)

In [57]:
df_population.shape

(10, 39)

In [52]:
df_population.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 39 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Whole country  9 non-null      str    
 1   Unnamed: 1     8 non-null      float64
 2   Unnamed: 2     8 non-null      float64
 3   Unnamed: 3     8 non-null      float64
 4   Unnamed: 4     8 non-null      float64
 5   Unnamed: 5     8 non-null      float64
 6   Unnamed: 6     8 non-null      float64
 7   Unnamed: 7     8 non-null      float64
 8   Unnamed: 8     8 non-null      float64
 9   Unnamed: 9     8 non-null      float64
 10  Unnamed: 10    8 non-null      float64
 11  Unnamed: 11    8 non-null      float64
 12  Unnamed: 12    8 non-null      float64
 13  Unnamed: 13    8 non-null      float64
 14  Unnamed: 14    8 non-null      float64
 15  Unnamed: 15    8 non-null      float64
 16  Unnamed: 16    8 non-null      float64
 17  Unnamed: 17    8 non-null      float64
 18  Unnamed: 18    8 non-nul

In [58]:
df_population.head()

,Whole country,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38
0,NaN,1987.0,1988.0,1989.0,1990.0,1991.0,1992.0,1993.0,1994.0,1995.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024.0
1,Employed,47.0,47.5,47.7,46.7,43.1,39.9,37.0,37.6,37.8,...,41.1,41.4,42.2,43.0,43.0,41.3,42.8,43.6,43.1,43.0
2,Unemployed,3.0,2.6,2.2,2.8,6.0,8.8,10.5,9.6,9.3,...,6.8,6.5,5.4,4.6,4.7,6.4,4.9,4.6,5.3,5.7
3,0-14 years old,19.3,19.4,19.3,19.3,19.2,19.2,19.1,19.1,19.0,...,16.3,16.2,16.2,16.0,15.8,15.6,15.4,15.1,14.9,14.6
4,"Students, pupils",6.3,6.2,6.4,6.6,7.1,7.4,8.4,8.7,8.9,...,7.5,7.4,7.3,7.1,7.3,7.6,7.6,7.4,7.6,7.9


In [34]:
df_population.isnull().sum()

Whole country    1
Unnamed: 1       2
Unnamed: 2       2
Unnamed: 3       2
Unnamed: 4       2
Unnamed: 5       2
Unnamed: 6       2
Unnamed: 7       2
Unnamed: 8       2
Unnamed: 9       2
Unnamed: 10      2
Unnamed: 11      2
Unnamed: 12      2
Unnamed: 13      2
Unnamed: 14      2
Unnamed: 15      2
Unnamed: 16      2
Unnamed: 17      2
Unnamed: 18      2
Unnamed: 19      2
Unnamed: 20      2
Unnamed: 21      2
Unnamed: 22      2
Unnamed: 23      2
Unnamed: 24      2
Unnamed: 25      2
Unnamed: 26      2
Unnamed: 27      2
Unnamed: 28      2
Unnamed: 29      2
Unnamed: 30      2
Unnamed: 31      2
Unnamed: 32      2
Unnamed: 33      2
Unnamed: 34      2
Unnamed: 35      2
Unnamed: 36      2
Unnamed: 37      2
Unnamed: 38      2
dtype: int64

In [ ]:
df_employment = pd.read_csv('../data/raw/statfin_employment_rate_monthly_2010_2026.csv')

In [ ]:
df_eurostat=pd.read_csv('../data/raw/eurostat_unemployment_monthly_eu_2025.csv')

In [ ]:
df_trends_yrittajaksi=pd.read_csv('../data/raw/google_trends_yrittajaksi_2019_2026.csv')

In [ ]:
df_trends_keikkayto=pd.read_csv('../data/raw/google_trends_keikkayto_2019_2026.csv')